# Choropleth Grammar: Classification, Legends, Small Multiples

A **choropleth** map shades polygons by a value. But the same data, colors, and class count under three classification schemes can tell three different stories about the same 428 Cuyahoga County census tracts. This session builds the vocabulary (`mapclassify`, `scheme=`, legends, small multiples) that form the foundation of mapping in Python.

By the end of this session you will be able to:

- Build a choropleth with `GeoDataFrame.plot(column=..., scheme=...)` and explain why that differs from calling `mapclassify` directly on a column with missing values
- Build a labeled static map step by step: classification, legend, missing data, frame, title, annotation, and a saved PNG
- Build a small-multiples figure comparing classification schemes on one column and describe what changes in the map's message
- Contrast a county-scale choropleth (428 tracts in one Ohio county) with a world-scale choropleth (177 countries)

In [ ]:
# Run this cell first (takes ~1 minute). Colab already includes geopandas.
%pip install -q libpysal esda splot mapclassify contextily
%pip install -q folium leafmap

## Let's talk about classification schemes

But first, what *is* a "classification scheme"?

What are some you're familiar with?

## Loading the tracts, and the gaps in them

`cuyahoga_tracts_acs.gpkg` pairs 428 Census-tract polygons with eight American Community Survey estimates (income, poverty, education, rent, home value, age, population). 

ACS suppresses an estimate whenever a tract's sample is too small to support it, so this file (like nearly all ACS products) contains `NaN` values. They are NOT a defect to fix before mapping - they are the state of the data, and the classification methods in this session must handle them correctly.

Note the file extension. This file is a GeoPackage rather than GeoJSON. `read_file` reads it the same way, and because the GeoPackage holds a **single layer** no `layer=` argument is needed. 

## Let's prepare our data

In [ ]:
DATA_BASE_P2 = "https://raw.githubusercontent.com/pjbitterman/KSU_x9076/main/data/part2"

# Local path fallback (uncomment if running outside Colab without internet access):
# DATA_BASE_P2 = "../data/part2"

import geopandas as gpd

tracts = gpd.read_file(f"{DATA_BASE_P2}/cuyahoga_tracts_acs.gpkg")
print(tracts.shape, tracts.crs)

# the columns we will use for the ACS analysis
acs_cols = ["median_hh_income", "pct_poverty", "pct_bachelors_plus", "pct_renter_occupied",
            "median_gross_rent", "median_home_value", "median_age", "total_population"]

# Check for missing values in the ACS columns
print(tracts[acs_cols].isna().sum())

# Check how many tracts have at least one missing ACS value
print("tracts with at least one missing ACS value:", tracts[acs_cols].isna().any(axis=1).sum())

# find the tract that is all water (GEOID 39035990000) and print its values
water_only = tracts[tracts["GEOID"] == "39035990000"]
print(water_only[["GEOID", "ALAND", "AWATER", "total_population", "median_hh_income"]])

**Expected result:** 49 of 428 tracts (11%) are missing at least one ACS value. 

But data gaps are not evenly distributed across columns.

Tract `39035990000` is a unusual case: `ALAND` is `0`, `AWATER` exceeds 2 billion square meters, `total_population` is `0` (a full count, not a missing value), and `median_hh_income` is `NaN` because no households live there. It is a stretch of open Lake Erie delineated as its own tract, with no ACS estimate to report. It is NOT a data error.

## `mapclassify` directly versus through `.plot()`

`mapclassify` performs the classification behind the `scheme=` argument of `.plot()`. Calling it directly on a column that still contains `NaN` values produces a result worth looking at BEFORE we use the safe pattern.

**Before running.** `tracts["median_hh_income"]` has 14 `NaN` values among 414 numeric ones. 

The next cell passes the column, `NaN` included, to `mapclassify.Quantiles(..., k=5)` without `.dropna()`. 

Do we expect `mapclassify` to raise an error, drop the `NaN` values and classify the remaining 414, or do something else?

In [ ]:
import mapclassify

raw_scheme = mapclassify.Quantiles(tracts["median_hh_income"], k=5)
print(raw_scheme)

**Expected result:** no error, and no silent drop. The first bin reports `[nan, 37297.60]` with **172** tracts, nearly double any other bin, and the last bin reports `(97029.80, nan]` with **zero**. `mapclassify` sorted the 14 `NaN` values among the numbers, and because `NaN` comparisons have their own place in numpy's sort order, every `NaN` row was placed in the lowest bin with the lowest incomes. That inflates the bin's count AND shifts the bin edges

## Building a map piece-by-piece

`.plot(column=..., scheme=...)` also calls `mapclassify`, but it classifies only the non-null rows and draws the null rows separately, with whatever `missing_kwds` is supplied. 

This is the safe pattern, and it is the one to use. 



### Step 1: the bare plot

`.plot(column=...)` is the whole choropleth in one argument. geopandas looks up each tract's value in the column and shades the polygon accordingly. 

**Before running.** The 14 tracts with no `median_hh_income` have nothing to look up. Will they appear in a default color, be skipped silently, or raise an error? And will the colors be a continuous ramp or a set of classes?

In [ ]:
import matplotlib.pyplot as plt

tracts.plot(column="median_hh_income")
plt.show()

**Step 1 outputs:** 

1. a smooth color ramp (matplotlib's default, viridis) with no legend
2. latitude and longitude tick labels around the edge
3. 14 white holes where the missing tracts were skipped without a warning. (The water tract is one of the 14, so the map shows land only)

A continuous ramp also means a reader can't tell which tracts fall in the same class, ***because there are no classes yet.***

### Step 2: classify

`scheme=` and `k=` hand the column to `mapclassify`, which bins the 414 non-null values into five classes. `cmap=` replaces viridis with `YlGnBu`, a sequential ColorBrewer palette designed for ordered classes.

In [ ]:
tracts.plot(
    column="median_hh_income",
    scheme="quantiles",   # NEW
    k=5,                  # NEW
    cmap="YlGnBu",        # NEW
)
plt.show()

**Step 2 outputs:** 
1. five discrete colors instead of a ramp
2. the pale and dark tracts now form legible blocks

But... nothing on the figure says what the five colors mean.

### Step 3: legend

`legend=True` asks `.plot()` to list the class breaks it used. NOTE where matplotlib places the legend, and compare the first break with the `37297.60` that the raw `mapclassify.Quantiles` call produced earlier.

In [ ]:
tracts.plot(
    column="median_hh_income",
    scheme="quantiles",
    k=5,
    cmap="YlGnBu",
    legend=True,          # NEW
)
plt.show()

**Step 3 outputs:** 
1. a legend with five ranges, the first running from `11036.00` to `36817.20`. 

Key difference: `.plot()` classified only the 414 non-null rows, so the `NaN` values that inflated the raw `mapclassify` result are not in the bins. 

Also... matplotlib placed the legend with `loc="best"`, in whatever corner had the most empty space (the upper right here).

The 14 missing tracts are still white holes, indistinguishable from the figure background, and the legend doesn't communicate about missing data.

### Step 4: show the missing data

`missing_kwds` tells `.plot()` how to draw the rows it set aside: a color, an outline, and a label that adds a sixth legend entry. Without this step, suppressed tracts "disappear" from the map and are not noted for the map reader.

**Before running.** Step 1's map showed land only because the water tract, with no `median_hh_income`, was skipped. The cell above reported its `AWATER` at over 2 billion square meters; the county's 427 land tracts total about 1.18 billion square meters of land. 

When `missing_kwds` draws the water tract in gray, what do we expect to happen to the map extent?

In [ ]:
tracts.plot(
    column="median_hh_income",
    scheme="quantiles",
    k=5,
    cmap="YlGnBu",
    legend=True,
    missing_kwds={"color": "lightgrey", "edgecolor": "white", "label": "no data (ACS suppressed)"},  # NEW
)
plt.show()

**Step 4 outputs:** 

- the 14 gray tracts appear and the legend gains a "no data" entry, but the map shifted substantially. 
- The water tract is 2,011 square kilometers of Lake Erie, nearly twice the county's 1,184 square kilometers of land, and matplotlib expanded the axes to fit it. 
The county now is less than half the size of a tall gray rectangle. 

**Data are correct, frame is wrong**

### Step 5: frame the map

Three changes. 

1. Create the figure and axes explicitly with `plt.subplots()`, so the figure size is under control and the axes can be passed to `.plot()` with `ax=`. 
2. Set the axis limits to the bounding box of the land tracts (`ALAND > 0`), with a small margin in degrees. 
3. Turn the axis frame off: latitude and longitude ticks provide little information on a map of a single county.

In [ ]:
land = tracts[tracts["ALAND"] > 0]            # NEW: the 427 land tracts
minx, miny, maxx, maxy = land.total_bounds    # NEW: get the bounding box of the land tracts

fig, ax = plt.subplots(figsize=(9, 7))        # NEW: create a figure and axes object
tracts.plot(
    column="median_hh_income",
    scheme="quantiles",
    k=5,
    cmap="YlGnBu",
    legend=True,
    missing_kwds={"color": "lightgrey", "edgecolor": "white", "label": "no data (ACS suppressed)"},
    ax=ax,                                    # NEW: plot on the axes object
)
ax.set_xlim(minx - 0.02, maxx + 0.02)         # NEW: set the x-axis limits to the bounding box of the land tracts, with a small buffer
ax.set_ylim(miny - 0.02, maxy + 0.02)         # NEW: set the y-axis limits to the bounding box of the land tracts, with a small buffer
ax.set_axis_off()                             # NEW: turn off the axes
plt.show()

**Step 5 outputs:** 
- the county now fills the frame
- water tract is reduced to a gray band along the shoreline
- gray band is still in the legend as "no data" 

***What do we still need to fix???***



### Step 6: title, legend, boundaries

1. `ax.set_title()` titles the map
2. `legend_kwds` provides the legend a title with units and pins it to the upper left, which is lake. 
3. `edgecolor` and `linewidth` draw thin white boundaries so that adjacent tracts in the same class remain distinguishable.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
tracts.plot(
    column="median_hh_income",
    scheme="quantiles",
    k=5,
    cmap="YlGnBu",
    legend=True,
    missing_kwds={"color": "lightgrey", "edgecolor": "white", "label": "no data (ACS suppressed)"},
    legend_kwds={"title": "Median household income ($)", "loc": "upper left", "fontsize": 8},  # NEW: legend title, location, and font size
    edgecolor="white",      # NEW: tract boundaries in white
    linewidth=0.2,          # NEW: tract boundary line width
    ax=ax,
)
ax.set_xlim(minx - 0.02, maxx + 0.02)
ax.set_ylim(miny - 0.02, maxy + 0.02)
ax.set_axis_off()
ax.set_title("Cuyahoga County median household income by tract\n(quantiles, k=5)")  # NEW: add a title to the map
plt.show()

**Step 6 outputs:** 
- a complete choropleth by the usual standard: classed colors, a titled legend with units, visible missing data, a title that discloses the classification. 

One thing a careful reader might ask about: the darkest class runs to `$250,001` - that's an oddly specific ceiling.

### Step 7: annotate and save

- `$250,001` is a Census top-code. ACS reports every tract median at or above `$250,000` as that single value, so the top of the range is a floor on the true figure, not a measurement. 

- `ax.annotate()` can say that on the map itself, with an arrow to the tract in question. 

- `.idxmax()` finds the first tract at the ceiling, and its centroid is then the target of an arrow. 

- The last line saves the figure as a PNG, which is how Sketch 5 asks you to hand in your maps alongside the notebook.

In [ ]:
# NEW: find the tract at the top-code and compute its centroid in a projected CRS
top = tracts.loc[[tracts["median_hh_income"].idxmax()]] # NEW: get the tract with the maximum median household income
print(top[["GEOID", "median_hh_income", "total_population"]]) # NEW: print the GEOID, median household income, and total population of the top-coded tract
top_pt = top.to_crs(26917).geometry.centroid.to_crs(4269).iloc[0] # NEW: compute the centroid of the top-coded tract in a projected CRS (EPSG:26917), then convert back to geographic CRS (EPSG:4269) and get the first point

fig, ax = plt.subplots(figsize=(9, 7))
tracts.plot(
    column="median_hh_income",
    scheme="quantiles",
    k=5,
    cmap="YlGnBu",
    legend=True,
    missing_kwds={"color": "lightgrey", "edgecolor": "white", "label": "no data (ACS suppressed)"},
    legend_kwds={"title": "Median household income ($)", "loc": "upper left", "fontsize": 8},
    edgecolor="white",
    linewidth=0.2,
    ax=ax,
)
ax.set_xlim(minx - 0.02, maxx + 0.02) # NOTE:, minx and miny are determined by an earlier cell. Know your environment and the order of execution of your cells. If you run this cell first, it will fail because minx and miny are not defined yet.
ax.set_ylim(miny - 0.02, maxy + 0.02) # same as above but for the y-axis
ax.set_axis_off()
ax.set_title("Cuyahoga County median household income by tract\n(quantiles, k=5)")
ax.annotate(                                                  # NEW: add an annotation for the top-coded tract
    "Highest tract: $250,001\n(Census top-code)",
    xy=(top_pt.x, top_pt.y),
    xytext=(top_pt.x - 0.28, top_pt.y + 0.14), # NEW: position the annotation text relative to the top-coded tract centroid
    arrowprops=dict(arrowstyle="->", color="black"), # add an arrow pointing to the top-coded tract
    fontsize=9,
)
fig.savefig("cuyahoga_income.png", dpi=200, bbox_inches="tight")  # NEW: save before plt.show()
plt.show()

**FINAL RESULT:** 

- First, calling `.centroid` directly on the `EPSG:4269` tracts would give the same `UserWarning` as an unreprojected buffer or area call, so the cell reprojects to `EPSG:26917` (meters), computes the centroid there, and converts the single resulting point back to `4269` for plotting. 
- NOTE: tract `39035179102` (population 2,641) is the first of two tracts reported at the `$250,001` ceiling; the other is `39035183200`, and `.idxmax()` returns whichever comes first in the table. The annotation sits over the lake with an arrow to the tract.

**MORE:**

- `fig.savefig()` writes `cuyahoga_income.png` to Colab's session storage; open the Files pane (folder icon at the left) to download it
- `dpi=200` gives a print-quality image and `bbox_inches="tight"` trims the surrounding whitespace. 

- NOTE: Keep the save BEFORE `plt.show()`: in a notebook, `show()` hands the figure to the display and closes it, so a `plt.savefig()` call in a later cell would write a new, blank figure.

### The whole recipe on a second variable

Once you're familiar with building the map, you can do the whole thing in a single cell. For example, here is  `pct_bachelors_plus`, with the annotation aimed at the highest-value tract.

In [ ]:
top_edu = tracts.loc[[tracts["pct_bachelors_plus"].idxmax()]]
print(top_edu[["GEOID", "pct_bachelors_plus", "total_population"]])
top_edu_pt = top_edu.to_crs(26917).geometry.centroid.to_crs(4269).iloc[0]

fig, ax = plt.subplots(figsize=(9, 7))
tracts.plot(
    column="pct_bachelors_plus",
    scheme="quantiles",
    k=5,
    cmap="PuBuGn",
    legend=True,
    missing_kwds={"color": "lightgrey", "edgecolor": "white", "label": "no data (ACS suppressed)"},
    legend_kwds={"title": "% with bachelor's degree+", "loc": "upper left", "fontsize": 8},
    edgecolor="white",
    linewidth=0.2,
    ax=ax,
)
ax.set_xlim(minx - 0.02, maxx + 0.02)
ax.set_ylim(miny - 0.02, maxy + 0.02)
ax.set_axis_off()
ax.set_title("Cuyahoga County educational attainment by tract\n(quantiles, k=5)")
ax.annotate(
    "Highest tract: 89.3%",
    xy=(top_edu_pt.x, top_edu_pt.y),
    xytext=(top_edu_pt.x - 0.18, top_edu_pt.y + 0.12),
    arrowprops=dict(arrowstyle="->", color="black"),
    fontsize=9,
)
plt.show()

**Exercise.** Build the full recipe for `median_home_value` (19 missing tracts). Choose a sequential colormap other than the two used above.

In [ ]:
### Try it here:



## Small multiples: one column, three classifications

`scheme="quantiles"` is just one option. 

`mapclassify`, through the `scheme=` argument, also supports **equal interval** (evenly spaced value ranges, not evenly sized bins) and **Fisher-Jenks** (breaks chosen to minimize within-group variance, often called natural breaks). 

If we place all three side by side on identical data, colors, and bin count, it demonstrates the importance of classification scheme

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 6))
schemes = ["quantiles", "equal_interval", "fisher_jenks"]
titles = ["Quantiles\n(equal count per bin)", "Equal interval\n(equal width per bin)", "Fisher-Jenks\n(natural breaks)"]

for ax, scheme, title in zip(axes, schemes, titles): # loop through the three classification schemes and plot each one
    tracts.plot(
        column="median_hh_income",
        scheme=scheme,
        k=5,
        cmap="YlGnBu",
        ax=ax,
        legend = True,
        missing_kwds={"color": "lightgrey", "edgecolor": "white"},
        legend_kwds={"title": "median HH income", "loc": "upper left", "fontsize": 8},
        edgecolor="white",
        linewidth=0.1,
    )
    ax.set_title(title, fontsize=11)
    ax.set_xlim(minx - 0.02, maxx + 0.02)
    ax.set_ylim(miny - 0.02, maxy + 0.02)
    ax.set_axis_off()

fig.suptitle("Median household income, three classification schemes", fontsize=13)
plt.tight_layout()
plt.show()

for scheme in schemes: # loop through the three classification schemes and print the counts for each one
    cls = getattr(mapclassify, {"quantiles": "Quantiles", "equal_interval": "EqualInterval", "fisher_jenks": "FisherJenks"}[scheme])
    print(scheme, list(cls(tracts["median_hh_income"].dropna(), k=5).counts))

**Expected result?** very uneven. 


### A second worked example: poverty rate

The equal-interval collapse on `median_hh_income` was driven by a top-coded ceiling stretching the value range. `pct_poverty` doesn't have that constraint

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 6))
for ax, scheme, title in zip(axes, schemes, titles):
    tracts.plot(
        column="pct_poverty",
        scheme=scheme,
        k=5,
        cmap="OrRd",
        ax=ax,
        legend = True,
        missing_kwds={"color": "lightgrey", "edgecolor": "white"},
        legend_kwds={"title": "% poverty", "loc": "upper left", "fontsize": 8},
        edgecolor="white",
        linewidth=0.1,
    )
    ax.set_title(title, fontsize=11)
    ax.set_xlim(minx - 0.02, maxx + 0.02)
    ax.set_ylim(miny - 0.02, maxy + 0.02)
    ax.set_axis_off()

fig.suptitle("% of families below the poverty line: same 428 tracts, three classification schemes", fontsize=13)
plt.tight_layout()
plt.show()

for scheme in schemes:
    cls = getattr(mapclassify, {"quantiles": "Quantiles", "equal_interval": "EqualInterval", "fisher_jenks": "FisherJenks"}[scheme])
    print(scheme, list(cls(tracts["pct_poverty"].dropna(), k=5).counts))

### What do you see?

**Exercise.** Run the same three-scheme comparison on `median_gross_rent` or `median_age`. Which scheme gives the most even bin counts for the variable, and is that what the variable's minimum, maximum, and mean would predict?

In [ ]:
### Try it here:



## A world map, for contrast of scale

So far we've been in Ohio.

`natural_earth_countries.geojson` contains 177 countries, the whole globe, and a single variable (population) rather than eight. The choropleth grammar stays the same though - but the scale changes quite a bit :)

### Let's try something new

In [ ]:
ne = gpd.read_file(f"{DATA_BASE_P2}/natural_earth_countries.geojson")
print(ne.shape, ne.crs)

valid = ne.geometry.is_valid # check the validity of each geometry in the GeoDataFrame
print(f"{valid.sum()} of {len(ne)} geometries are valid")
print(ne.loc[~valid, ['name']])

**Expected result:** 176 of 177 valid. **Sudan** is the exception, a self-intersecting polygon present in Natural Earth's own 1:110m cultural data. We're not going to talk about geometry repair for now. 

But it doesn't break the choropleth below, since `.plot()` and `mapclassify` tolerate it, so no repair is applied.

If a later operation requires valid geometry, such as `gpd.overlay()`, `geometry.buffer(0)` is the standard repair: a zero-distance buffer rebuilds the polygon's rings without changing its shape. 

Checking `.is_valid` on a new file is the geometric counterpart of checking `.isna()` on a column.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 6))
ne.plot(
    column="pop_est",
    scheme="quantiles",
    k=5,
    cmap="YlOrRd",
    legend=True,
    ax=ax,
    legend_kwds={"title": "Population (est.)", "loc": "lower left", "fontsize": 8},
)
ax.set_title("World population by country (quantiles, k=5): 177 countries, one map, no missing data")
ax.set_axis_off()
plt.tight_layout()
plt.show()

print(ne.sort_values("pop_est", ascending=False)[["name", "pop_est"]].head(5))

**Expected result:** a five-color world map with quantile bins of 36/35/35/35/36 countries. 

`pop_est` has no missing values, so `missing_kwds` is not needed here. 

China (1.40 billion) and India (1.37 billion) lead, each alone exceeding dozens of the smallest countries combined, a range of variation far wider than `median_hh_income`'s.

In [ ]:
# Does equal interval collapse at this scale as it did on the tracts?
eq = mapclassify.EqualInterval(ne["pop_est"], k=5)
print(eq)

**Expected result:** an even MORE extreme than either tract example: `174, 1, 0, 0, 2`. China and India occupy the top bin, the United States (328 million) sits alone in a bin of its own, two middle bins are empty, and 174 of 177 countries fall into the lowest bin.

 Population is among the most skewed variables at the country scale, and equal interval, which treats equal width as fair regardless of the distribution, fails here as it did on county income and poverty, more visibly because the outliers are larger relative to the rest.

**Exercise.** Re-run the world map with `column="gdp_md"` (GDP in millions of current US dollars) instead of `pop_est`, keeping `scheme="quantiles"`. Do the same five countries lead the GDP ranking as led the population ranking?

In [ ]:
## Try it here:



## Summary

One principle in three settings: **the classification scheme determiens which polygons appear high and which appear low, independent of the underlying values.**

Quantiles guarantees an even split but can separate similar values at a bin edge. Equal interval appears fair but collapses whenever a distribution has outliers, which every ACS variable and Natural Earth's population column do. 

Fisher-Jenks follows the data's own groupings at the cost of being harder to explain to a reader. 

***NONE IS CORRECT IN THE ABSTRACT***

The defensible practice is to choose deliberately, disclose the choice, and know what a different choice would have shown. Sketch 5 asks you to demonstrate exactly that.

## Wrap-up and next steps

**Practice** (no submission):

1. Build a `.plot(column=..., scheme="fisher_jenks", k=5, missing_kwds=...)` choropleth of `median_age` with a title and legend. Which tracts appear extreme, and does that match your expectation of where younger and older populations cluster in Cuyahoga County?
2. Change `k` from `5` to `3` in any of the small-multiples cells. Does equal interval's collapse improve, worsen, or stay the same with fewer bins?
3. On the world map, filter `ne` to `continent == "Africa"` (or another continent) before plotting a `pop_est` quantile choropleth, a scale between county and world. How many countries remain in the subset?

**This week's reading:** ["Choropleth Mapping"](https://geographicdata.science/book/notebooks/05_choropleth.html) (*Geographic Data Science with Python*, chapter 5), which covers the same quantiles, equal-interval, and Fisher-Jenks vocabulary with additional schemes. ["Getting started"](https://python-visualization.github.io/folium/latest/getting_started.html) (Folium documentation) previews the next session.


**Next session:** interactive maps with folium and leafmap: points, popups, and a clickable web choropleth. 